# Dataset Audit Notebook
This notebook performs a systematic audit of all three breast ultrasound datasets used in this thesis: BUSI (Egypt), BUS-UCLM (Spain), and BrEaST (Poland).

The audit covers image counts, colour mode analysis, class distribution, resolution variability, metadata integration, annotation artefact characterisation, and duplicate detection.

## 1. Mount Drive
Mount Google Drive, import libraries, and define dataset paths.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Import Libraries
Standard libraries for file handling (`os`, `pathlib`), data manipulation (`pandas`), image processing (`PIL`, `numpy`), and visualisation (`matplotlib`).

No GPU-dependent libraries are required for this audit notebook.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

### Dependencies
Installing any libraries not available by default in Colab.

In [ ]:
!pip install openpyxl -q

## 3. Define Paths
All dataset paths are defined relative to the thesis root directory on Google Drive.

Centralising paths here means a single update propagates throughout the entire notebook if the folder structure changes.

In [ ]:
# Root path
ROOT = Path('/content/drive/MyDrive/University/Masters/MSc_Thesis')

# Raw data paths
RAW = ROOT / 'data' / 'raw'

BUSI_PATH = RAW / 'BUSI' / 'Dataset_BUSI' / 'Dataset_BUSI_with_GT'
BUSUCLM_PATH = RAW / 'BUS-UCLM' / 'BUS-UCLM Breast ultrasound lesion segmentation dataset' / 'BUS-UCLM' / 'images'
BREAST_PATH = RAW / 'BrEaST' / 'BrEaST-Lesions' / 'BrEaST-Lesions_USG-images_and_masks'

# Verify paths exist
for name, path in [('BUSI', BUSI_PATH), ('BUS-UCLM', BUSUCLM_PATH), ('BrEaST', BREAST_PATH)]:
  print(f'{name}: {"Found" if path.exists() else "NOT FOUND"}')

## 4. Dataset Scanning
The `scan_dataset` function recursively searches each dataset folder for PNG images, excluding masks by filename. For each image it records the filename, full filepath, parent folder class label, colour mode, dimensions, and file size.

Note: Class labels derived from folder names are only valid for BUSI.

BUS-UCLM and BrEaST class labels are overwritten during metadata integration in Section 5.

In [ ]:
def scan_dataset(dataset_path, dataset_name):
  records = []
  exclude = ['mask', 'tumor', 'other']
  for image_path in dataset_path.rglob('*.png'):
    if not any(word in image_path.name.lower() for word in exclude):
      img = Image.open(image_path)
      records.append({
          'dataset': dataset_name,
          'image_name': image_path.name,
          'filepath': str(image_path),
          'class': image_path.parent.name,
          'color_mode': img.mode,
          'width': img.size[0],
          'height': img.size[1],
          'file_size_kb': round(image_path.stat().st_size / 1024, 2)
      })
  return records

### Build Master Dataframe
Scan all three datasets and combine into a single master dataframe.

Expected total: 1719 images (780 BUSI + 683 BUS-UCLM + 256 BrEaST).

In [ ]:
all_records = []
all_records.extend(scan_dataset(BUSI_PATH, 'BUSI'))
all_records.extend(scan_dataset(BUSUCLM_PATH, 'BUS-UCLM'))
all_records.extend(scan_dataset(BREAST_PATH, 'BrEaST'))

df = pd.DataFrame(all_records)
print(f'Total images found: {len(df)}')
df.head(10)

## 5. Metadata Integration
BUS-UCLM and BrEaST provide class labels and additional metadata via CSV and Excel files, respectively. These are merged into the master dataframe using image filename as the key, replacing the placeholder folder-name class labels assigned during scanning.

### BUS-UCLM Metadata Integration
BUS-UCLM metadata is provided as a semicolon-delimited CSV file.

Key columns merged into the master dataframe:
`Label` (class label), `Doppler`, `Marks`, and `Combined` (acquisition and annotation flags stored as Yes/No strings).

In [ ]:
# Load BUS-UCLM metadata
busuclm_meta = pd.read_csv(BUSUCLM_PATH.parent / 'INFO.csv', delimiter = ';')

print(busuclm_meta.head())

print(busuclm_meta.columns.tolist())

In [ ]:
# Merge BUS-UCLM into master dataframe
cols_to_merge = ['Image', 'Label', 'Doppler', 'Marks', 'Combined']
busuclm_merge = busuclm_meta[cols_to_merge].rename(
    columns = {'Image': 'image_name', 'Label': 'class'}
).set_index('image_name')

for col in ['class', 'Doppler', 'Marks', 'Combined']:
  df.loc[df['dataset'] == 'BUS-UCLM', col] = (
      df.loc[df['dataset'] == 'BUS-UCLM', 'image_name'].map(busuclm_merge[col])
  )

print(df[df['dataset'] == 'BUS-UCLM'][
    ['image_name', 'class', 'Doppler', 'Marks', 'Combined']
].head(10))

### BrEaST Metadata Integration
BrEaST clinical data is provided as an Excel file.

Key columns merged into the master dataframe:
`Classification` (class label), `Pixel_size` (physical scale in mm/pixel), and `Mask_other_filename` (presence of additional suspicious lesions, stored as boolean `has_other_lesion`).

In [ ]:
# Load BrEaST metadata
breast_meta = pd.read_excel(BREAST_PATH.parent.parent / 'BrEaST-Lesions-USG-clinical-data.xlsx')

print(breast_meta.head())

print(breast_meta.columns.tolist())

In [ ]:
cols_to_merge = ['Image_filename', 'Classification', 'Pixel_size', 'Mask_other_filename']
breast_merge = breast_meta[cols_to_merge].rename(
    columns = {'Image_filename': 'image_name', 'Classification': 'class'}
).set_index('image_name')

for col in ['class', 'Pixel_size']:
  df.loc[df['dataset'] == 'BrEaST', col] = (
      df.loc[df['dataset'] == 'BrEaST', 'image_name'].map(breast_merge[col])
  )

df.loc[df['dataset'] == 'BrEaST', 'has_other_lesion'] = df.loc[
    df['dataset'] == 'BrEaST', 'image_name'].map(
    breast_merge['Mask_other_filename'].notna())

print(df[df['dataset'] == 'BrEaST'][
    ['image_name', 'class', 'Pixel_size', 'has_other_lesion']
].head(10))

## 6. Dataset Characteristics
Analysis of class distribution and resolution variability across all three datasets.

### Class Distribution
Verified against published dataset statistics.

BUSI: 437 benign, 210 malignant, 133 normal

BUS-UCLM: 174 benign, 90 malignant, 419 normal

BrEaST: 154 benign, 98 malignant, 4 normal

In [ ]:
print(df.groupby(['dataset', 'class'])['image_name'].count())

### Class Imbalance Summary
Explicit imbalance ratios across all three datasets.

Informs loss function selection, augmentation strategy, and metric interpretation in subsequent notebooks.

**Note**: BUSI counts reflect pre-exclusion numbers.

Final counts pending radiologist review and application of BrEaST authors' exclusion list.

### Colour Mode Analysis
Two-step analysis of colour content across all three datasets.

Step 1: Declared colour mode per dataset (RGB, RGBA, or L) as stored in the image file.

Step 2: Genuine colour detection -- grayscale images saved as RGB have identical R, G, and B channel values. Images with differing channel values contain real colour content (Doppler signals, scanner UI elements, annotation artefacts).

In [ ]:
print(df.groupby(['dataset', 'color_mode'])['image_name'].count())

In [ ]:
def is_truly_rgb(image_path):
  img = Image.open(image_path).convert('RGB')
  r, g, b = img.split()
  return not (list(r.getdata()) == list(g.getdata()) == list(b.getdata()))

from pathlib import Path

truly_rgb = []
for idx, row in df.iterrows():
  img_path = Path(row['filepath'])
  if row['color_mode'] in ['RGB', 'RGBA']:
    if is_truly_rgb(img_path):
      truly_rgb.append(row['image_name'])

print(f'Images with genuine colour content: {len(truly_rgb)}')

#### Genuine Colour Detection Function
Converts each image to RGB and compares channel values pixel by pixel.

Returns `True` if channels differ, indicating genuine colour content.

In [ ]:
df['is_truly_rgb'] = False

for idx, row in df.iterrows():
  if row['color_mode'] in ['RGB', 'RGBA']:
    if row['image_name'] in truly_rgb:
      df.at[idx, 'is_truly_rgb'] = True

print(df.groupby(['dataset', 'is_truly_rgb'])['image_name'].count())

In [ ]:
print(df.groupby(['dataset', 'class', 'is_truly_rgb'])['image_name'].count())

### BUSI Colour Image Investigation
16 BUSI images flagged as genuinely colour. Isolated and manually inspected.

Findings: a mix of true Doppler signal, yellow bounding boxes, callipers, poorly cropped scanner edges, and one ambiguous case.

Class distribution: 12 benign, 3 malignant, 1 normal.

Note: the single colour normal image is particularly sensitive given BUSI's already small normal class (133 images).

In [ ]:
busi_rgb = df[(df['dataset'] == 'BUSI') & (df['is_truly_rgb'] == True)]

print(busi_rgb[['image_name', 'class', 'color_mode']])

In [ ]:
print(busi_rgb.groupby('class')['image_name'].count())

### Resolution Distribution
Variability in image dimensions across datasets directly informs the resizing strategy in preprocessing.

All images will be resized to a standard input dimension during preprocessing.

In [ ]:
print("=== Resolution Distribution ===")

for dataset in ['BUSI', 'BUS-UCLM', 'BrEaST']:
    subset = df[df['dataset'] == dataset]

    print(f"\n{dataset}:")
    print(f"  Width  — min: {subset['width'].min()}, max: {subset['width'].max()}, median: {subset['width'].median()}")
    print(f"  Height — min: {subset['height'].min()}, max: {subset['height'].max()}, median: {subset['height'].median()}")
    print(f"  Unique resolutions: {subset.groupby(['width','height']).ngroups}")

## 7. Image Quality Checks
Systematic verification of image integrity across all 1719 images.

Checks: corrupt or unreadable files, blank or near-blank images, and duplicate detection.

### Corrupt or Unreadable Images
Attempts to open and verify every image using PIL's `verify()` method.

A `try/except` block catches any image that fails to open cleanly.

Result: 0 corrupt or unreadable images across all three datasets.

In [ ]:
corrupt = []

for idx, row in df.iterrows():
    try:
        img = Image.open(row['filepath'])
        img.verify()
    except Exception as e:
        corrupt.append({
            'image_name': row['image_name'],
            'dataset': row['dataset'],
            'error': str(e)
        })

print(f"Corrupt or unreadable images: {len(corrupt)}")
if corrupt:
    print(pd.DataFrame(corrupt))

### Blank or Near-Blank Images
Images with pixel standard deviation below 10 are flagged as blank or near-blank, indicating no diagnostic content.

Conversion to grayscale (mode L) ensures consistent comparison across RGB and RGBA images.

Result: 0 blank or near-blank images across all three datasets.

In [ ]:
blank = []

for idx, row in df.iterrows():
    img = Image.open(row['filepath']).convert('L')
    arr = np.array(img)
    if arr.std() < 10:
        blank.append({
            'image_name': row['image_name'],
            'dataset': row['dataset'],
            'class': row['class'],
            'std': round(arr.std(), 2)
        })

print(f"Blank or near-blank images: {len(blank)}")
if blank:
    print(pd.DataFrame(blank))

### Duplicate Detection
Image content is compared using MD5 hashing -- identical images produce identical hash regardless of filename. Applied to BUSI only, as it is the training dataset where duplicates directly compromise model evaluation integrity.

Result: 1 duplicate pair found across benign and malignant classes -- same image, different labels. Flagged for radiologist review to determine correct label.

In [ ]:
import hashlib

def get_image_hash(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

busi_df = df[df['dataset'] == 'BUSI'].copy()
busi_df['image_hash'] = busi_df['filepath'].apply(get_image_hash)

duplicates = busi_df[busi_df.duplicated(subset='image_hash', keep=False)]
print(f"Duplicate images found in BUSI: {len(duplicates)}")
print(duplicates[['image_name', 'class', 'image_hash']])

## 8. Annotation Artefact Analysis
Characterisation of annotation artefacts, label consistency, and additional metadata flags across all three datasets.

### Class Label Consistency Check
Verifies that no unexpected label values exist beyond normal, benign, and malignant.

Capitalisation inconsistency is solved in the preprocessing stage.

In [ ]:
print("=== Unique class labels per dataset ===")

for dataset in ['BUSI', 'BUS-UCLM', 'BrEaST']:
    labels = df[df['dataset'] == dataset]['class'].unique()
    print(f"\n{dataset}: {labels}")

### BUS-UCLM Annotation Flags
Quantification of Doppler, Marks, and Combined flags from metadata.

BUS-UCLM is blind external validation only -- these flags inform result interpretation rather than preprocessing exclusion decisions.

In [ ]:
busuclm_df = df[df['dataset'] == 'BUS-UCLM']

print("=== BUS-UCLM Annotation Flags ===")
print(f"Doppler: {(busuclm_df['Doppler'] == 'Yes').sum()}")
print(f"Marks: {(busuclm_df['Marks'] == 'Yes').sum()}")
print(f"Combined: {(busuclm_df['Combined'] == 'Yes').sum()}")

print("\n=== Doppler by class ===")
print(busuclm_df[busuclm_df['Doppler'] == 'Yes'].groupby('class')['image_name'].count())

print("\n=== Marks by class ===")
print(busuclm_df[busuclm_df['Marks'] == 'Yes'].groupby('class')['image_name'].count())

print("\n=== Combined by class ===")
print(busuclm_df[busuclm_df['Combined'] == 'Yes'].groupby('class')['image_name'].count())

### BrEaST Additional Suspicious Lesions
7 images contain masks of additional suspicious lesions beyond the primary labelled finding. These images are flagged for radiologist attention during GradCAM review in notebook 05, as model attention may reflect a real, but unlabelled lesion rather than a classification error.

In [ ]:
print(df['has_other_lesion'].value_counts())

## 9. Audit Summary and Export
Summary of all findings from the dataset audit. The master dataframe is exported to CSV for reference and shared with the audit spreadsheet in Google Sheets.

### Key Findings
**Dataset composition:**
- Total images: 1719 (780 BUSI, 683 BUS-UCLM, 256 BrEaST)
- All image counts verified against published dataset statistics.

**Image quality:**
- 0 corrupt or unreadable images
- 0 blank or near-blank images
- 1 mislabelled duplicate pair in BUSI (benign/malignant) -- flagged for radiologist review

**Colour content:**
- BUSI: 16 genuinely colour images -- Doppler, callipers, scanner edges
- BUS-UCLM: all 683 images contain scanner UI elements (yellow logo, BIRADS diagram)
- BrEaST: 163 images flagged -- requires visual verification

**Resolution variability:**
- BUSI: extreme variability (639 unique resolutions)
- BUS-UCLM: highly consistent (13 unique resolutions)
- BrEaST: moderate variability (52 unique resolutions)

**Annotation artefacts:**
- BUSI: 102 benign images with measuring lines
- BUS-UCLM: 36 Doppler, 154 Marks, 6 Combined -- documented for result interpretation
- BrEaST: 7 images with additional suspicious lesions -- flagged for radiologist review

**Label consistency:**
- Capitalisation inconsistency confirmed across datasets
- No unexpected label values found

### Post-Audit Exclusions (Applied in Notebook 02)
Confirmed exclusions from BUSI following BrEaST authors' critique:
- Axilla images: 70 (not breast tissue)
- Needle images: 7 (post-procedure artefact)
- Pictogram overlap with axilla: 1 (already counted)
- Total confirmed exclusions: 77
- Remaining BUSI images: 703

### Pending Radiologist Review
19 images flagged for radiologist label verification:
- Near-duplicated series: radiologist selects highest quality image
- Multiclass images: radiologist assigns correct label
- MD5 duplicate pair: radiologist assigns correct label
- Radiologist document prepared separately
- Decisions applied as first step in Notebook 02


In [ ]:
# Export master dataframe to CSV
output_path = ROOT / 'outputs' / 'results' / 'audit_master.csv'
df.to_csv(output_path, index = False)

print(f"Master dataframe exported to: {output_path}")
print(f"Shape: {df.shape}")